# Fiorell.IA LoRA Cloud Training\n\nThis notebook runs Fiorell.IA behavior tuning in Colab without changing the shared backend runtime.\n\nSupported paths:\n- clone the repo from GitHub;\n- use an existing copy in Google Drive.\n\nRecommended runtime: Colab with GPU.\n

## 1. Configure The Run\n\nSet `REPO_MODE` before running the next cells:\n- `github_public`\n- `github_private`\n- `drive_existing`\n

In [ ]:
from pathlib import Path\n\nREPO_MODE = "github_public"\nREPO_URL = "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git"\nREPO_BRANCH = "main"\nREPO_DIR = Path("/content/regulatory-insight-engine")\nDRIVE_REPO_DIR = Path("/content/drive/MyDrive/regulatory-insight-engine")\nDRIVE_ARTIFACTS_DIR = Path("/content/drive/MyDrive/fiorellia-runs")\nCONFIG_PATH = "fiorellia/training/configs/config_lora_behavior_20260421.yaml"\nENABLE_DRIVE_EXPORT = True\nprint({\n    "REPO_MODE": REPO_MODE,\n    "REPO_URL": REPO_URL,\n    "REPO_BRANCH": REPO_BRANCH,\n    "REPO_DIR": str(REPO_DIR),\n    "DRIVE_REPO_DIR": str(DRIVE_REPO_DIR),\n    "DRIVE_ARTIFACTS_DIR": str(DRIVE_ARTIFACTS_DIR),\n    "CONFIG_PATH": CONFIG_PATH,\n    "ENABLE_DRIVE_EXPORT": ENABLE_DRIVE_EXPORT,\n})\n

## 2. Optional: Mount Google Drive\n\nRun this if you want to keep outputs after the Colab session ends.\n

In [ ]:
from google.colab import drive\n\ndrive.mount('/content/drive')\n

## 3. Prepare The Repository\n\nFor a private repo, the cell will ask for a temporary GitHub token and use it only for the clone command.\n

In [ ]:
import os\nimport shutil\nimport subprocess\nfrom getpass import getpass\n\nif REPO_MODE in {"github_public", "github_private"}:\n    if REPO_DIR.exists():\n        shutil.rmtree(REPO_DIR)\n    clone_url = REPO_URL\n    if REPO_MODE == "github_private":\n        token = getpass("GitHub token (temporary, input hidden): ").strip()\n        if not token:\n            raise ValueError("A token is required for github_private mode.")\n        clone_url = REPO_URL.replace("https://", f"https://{token}@")\n    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)], check=True)\nelif REPO_MODE == "drive_existing":\n    if not DRIVE_REPO_DIR.exists():\n        raise FileNotFoundError(f"Drive repo not found: {DRIVE_REPO_DIR}")\n    if REPO_DIR.exists():\n        shutil.rmtree(REPO_DIR)\n    shutil.copytree(DRIVE_REPO_DIR, REPO_DIR)\nelse:\n    raise ValueError(f"Unsupported REPO_MODE: {REPO_MODE}")\n\nos.chdir(REPO_DIR)\nprint(f"repo_ready={REPO_DIR}")\n

## 4. Install Dependencies\n\nColab already provides PyTorch. This cell upgrades the Python stack needed by the Fiorell.IA training script.\n

In [ ]:
!python -m pip install --upgrade pip\n!pip install -r fiorellia/training/requirements-lora.txt\n!pip install bitsandbytes\n

## 5. Inspect Environment, Config, And Dataset\n

In [ ]:
import json\nfrom pathlib import Path\n\nimport torch\nimport yaml\n\nrepo_root = Path.cwd()\nconfig_path = repo_root / CONFIG_PATH\nconfig = yaml.safe_load(config_path.read_text(encoding="utf-8"))\ndataset_path = repo_root / config["dataset_path"]\noutput_dir = repo_root / config["output_dir"]\n\nprint(json.dumps({\n    "python": os.popen("python --version").read().strip(),\n    "cuda_available": torch.cuda.is_available(),\n    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n    "config_path": str(config_path),\n    "dataset_exists": dataset_path.exists(),\n    "dataset_path": str(dataset_path),\n    "output_dir": str(output_dir),\n    "base_model_name": config["base_model_name"],\n    "use_4bit": config.get("use_4bit"),\n}, indent=2))\n\nwith dataset_path.open("r", encoding="utf-8") as handle:\n    sample = json.loads(next(line for line in handle if line.strip()))\nprint(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])\n

## 6. Optional: Quick Preflight\n\nThis reuses the existing project preflight.\n

In [ ]:
!python fiorellia/training/preflight_check_training.py\n

## 7. Start Training\n\nThis uses the existing Fiorell.IA training script and dated config.\n

In [ ]:
!python fiorellia/training/train_lora_behavior_v1.py --config "$CONFIG_PATH"\n

## 8. Export Adapter Artifacts\n\nThis copies the final adapter to a stable export folder and prepares a zip for download.\n

In [ ]:
import shutil\n\nartifact_name = output_dir.name\nartifact_root = repo_root / "artifacts"\nartifact_root.mkdir(parents=True, exist_ok=True)\nartifact_dir = artifact_root / artifact_name\nif artifact_dir.exists():\n    shutil.rmtree(artifact_dir)\nshutil.copytree(output_dir, artifact_dir)\nzip_base = artifact_root / artifact_name\nzip_path = shutil.make_archive(str(zip_base), "zip", root_dir=artifact_root, base_dir=artifact_name)\nprint(f"artifact_dir={artifact_dir}")\nprint(f"zip_path={zip_path}")\n\nif ENABLE_DRIVE_EXPORT:\n    DRIVE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)\n    drive_artifact_dir = DRIVE_ARTIFACTS_DIR / artifact_name\n    if drive_artifact_dir.exists():\n        shutil.rmtree(drive_artifact_dir)\n    shutil.copytree(artifact_dir, drive_artifact_dir)\n    drive_zip = shutil.make_archive(str(DRIVE_ARTIFACTS_DIR / artifact_name), "zip", root_dir=DRIVE_ARTIFACTS_DIR, base_dir=artifact_name)\n    print(f"drive_artifact_dir={drive_artifact_dir}")\n    print(f"drive_zip={drive_zip}")\n

## 9. Download The Zip\n\nUse this only if you want to download directly from the Colab session.\n

In [ ]:
from google.colab import files\n\nfiles.download(str(artifact_root / f"{artifact_name}.zip"))\n

## 10. Next Local Step\n\nPut the adapter back under:\n\n```\nfiorellia/training/lora/fiorellia_behavior_20260421/\n```\n\nThen follow:\n- `fiorellia/eval/prompt_harness_with_adapter.md`\n- `fiorellia/eval/compare_baseline_vs_adapter_20260421.md`\n